# Foundry + OpenTelemetry + LangSmith — Step-by-Step

This notebook walks through each component individually so you can verify the setup works before running end-to-end.

## 1. Install Dependencies

In [ ]:
%pip install azure-ai-projects azure-ai-agents azure-identity azure-core-tracing-opentelemetry opentelemetry-sdk opentelemetry-exporter-otlp-proto-http python-dotenv --quiet

## 2. Define Environment Variables

Fill in your values below. These stay in-memory (not written to disk).

In [ ]:
import os

# === FILL THESE IN ===
os.environ["PROJECT_ENDPOINT"] = "https://your-project.services.ai.azure.com"  # Azure AI Foundry endpoint
os.environ["MODEL_DEPLOYMENT_NAME"] = "gpt-4o"  # Your model deployment name
os.environ["LANGSMITH_OTEL_ENDPOINT"] = "https://api.smith.langchain.com/otel/v1/traces"  # LangSmith OTLP endpoint
os.environ["LANGSMITH_API_KEY"] = "lsv2_your_api_key_here"  # LangSmith API key
os.environ["OTEL_SERVICE_NAME"] = "foundry-otel-langsmith"
os.environ["DEPLOYMENT_ENVIRONMENT"] = "development"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"  # Enable for debugging

print("✅ Environment variables set")

## 3. Test Azure Authentication

Verify you can authenticate to Azure before doing anything else.

In [ ]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
token = credential.get_token("https://management.azure.com/.default")
print(f"✅ Authenticated successfully (token expires: {token.expires_on})")

## 4. Test AIProjectClient Connection

Verify the Foundry project endpoint is reachable and the agents API is accessible.

In [ ]:
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

with project_client:
    agents_client = project_client.agents
    print(f"✅ Connected to project: {os.environ['PROJECT_ENDPOINT']}")
    print(f"   Agents client type: {type(agents_client).__name__}")

## 5. Test OpenTelemetry Setup (Console Export)

Before routing to LangSmith, verify OTEL spans are created correctly by exporting to console.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource

# Set up a console exporter to see spans printed here
resource = Resource.create({"service.name": "notebook-test"})
console_provider = TracerProvider(resource=resource)
console_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(console_provider)

# Create a test span
tracer = trace.get_tracer("test")
with tracer.start_as_current_span("test.hello") as span:
    span.set_attribute("test.message", "Hello from notebook!")
    print("\n✅ If you see a span JSON object above/below, OTEL is working!")

console_provider.shutdown()

## 6. Test OTLP Export to LangSmith

Send a test span to LangSmith and verify it arrives. Check your LangSmith dashboard after running this.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

resource = Resource.create({
    "service.name": os.environ["OTEL_SERVICE_NAME"],
    "deployment.environment": os.environ["DEPLOYMENT_ENVIRONMENT"],
})

langsmith_provider = TracerProvider(resource=resource)
langsmith_exporter = OTLPSpanExporter(
    endpoint=os.environ["LANGSMITH_OTEL_ENDPOINT"],
    headers={"x-api-key": os.environ["LANGSMITH_API_KEY"]},
)
langsmith_provider.add_span_processor(SimpleSpanProcessor(langsmith_exporter))
trace.set_tracer_provider(langsmith_provider)

tracer = trace.get_tracer("langsmith-test")
with tracer.start_as_current_span("notebook.test_export") as span:
    span.set_attribute("test.source", "jupyter-notebook")
    span.set_attribute("test.step", "verify_langsmith_connection")

langsmith_provider.shutdown()
print("✅ Test span sent to LangSmith!")
print("   → Check your LangSmith dashboard for a 'notebook.test_export' trace.")

## 7. Test AIAgentsInstrumentor

Verify the SDK's built-in instrumentor hooks into agent operations.

In [ ]:
from azure.core.settings import settings
from azure.ai.agents.telemetry import AIAgentsInstrumentor
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory import InMemorySpanExporter
from opentelemetry.sdk.resources import Resource

# Enable azure-core OTEL bridge
settings.tracing_implementation = "opentelemetry"

# Use in-memory exporter to inspect spans locally
memory_exporter = InMemorySpanExporter()
resource = Resource.create({"service.name": "instrumentor-test"})
test_provider = TracerProvider(resource=resource)
test_provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
trace.set_tracer_provider(test_provider)

# Instrument the agents SDK
AIAgentsInstrumentor().instrument()
print("✅ AIAgentsInstrumentor activated")
print("   All agent SDK calls will now emit OTEL spans automatically.")

## 8. Run the Agent (End-to-End with Tracing)

This creates a code interpreter agent, sends a prompt, and traces everything to LangSmith.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from azure.core.settings import settings
from azure.ai.agents.telemetry import AIAgentsInstrumentor
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import CodeInterpreterTool, ListSortOrder, MessageRole
from azure.identity import DefaultAzureCredential

# --- Telemetry setup ---
settings.tracing_implementation = "opentelemetry"

resource = Resource.create({
    "service.name": os.environ["OTEL_SERVICE_NAME"],
    "deployment.environment": os.environ["DEPLOYMENT_ENVIRONMENT"],
})
provider = TracerProvider(resource=resource)
exporter = OTLPSpanExporter(
    endpoint=os.environ["LANGSMITH_OTEL_ENDPOINT"],
    headers={"x-api-key": os.environ["LANGSMITH_API_KEY"]},
)
provider.add_span_processor(BatchSpanProcessor(exporter))
trace.set_tracer_provider(provider)
AIAgentsInstrumentor().instrument()

tracer = trace.get_tracer(__name__)
print("✅ Telemetry initialized — traces will export to LangSmith")

In [ ]:
# --- Run the agent ---
PROMPT = "Write a Python function that calculates the first 20 Fibonacci numbers, then print them as a formatted table."

with tracer.start_as_current_span("notebook.end_to_end_test") as root_span:
    root_span.set_attribute("prompt", PROMPT)

    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    code_interpreter = CodeInterpreterTool()

    with project_client:
        agents_client = project_client.agents

        # Create agent
        agent = agents_client.create_agent(
            model=os.environ["MODEL_DEPLOYMENT_NAME"],
            name="notebook-code-interpreter",
            instructions="You are a helpful Python coding assistant. Write and execute code to answer questions.",
            tools=code_interpreter.definitions,
        )
        print(f"✅ Agent created: {agent.id}")

        try:
            # Create thread
            thread = agents_client.threads.create()
            print(f"✅ Thread created: {thread.id}")

            # Send message
            message = agents_client.messages.create(
                thread_id=thread.id,
                role="user",
                content=PROMPT,
            )
            print(f"✅ Message sent: {message.id}")

            # Run agent
            print("⏳ Running agent...")
            run = agents_client.runs.create_and_process(
                thread_id=thread.id, agent_id=agent.id
            )
            print(f"✅ Run completed with status: {run.status}")
            root_span.set_attribute("run.status", run.status)

            if run.status == "failed":
                print(f"❌ Error: {run.last_error}")
            else:
                # Get response
                messages = agents_client.messages.list(
                    thread_id=thread.id, order=ListSortOrder.DESCENDING
                )
                for msg in messages:
                    if msg.role == MessageRole.AGENT and msg.text_messages:
                        response = msg.text_messages[-1].text.value
                        print(f"\n{'='*60}")
                        print("AGENT RESPONSE:")
                        print(f"{'='*60}")
                        print(response)
                        break
        finally:
            agents_client.delete_agent(agent.id)
            print(f"\n🧹 Agent deleted")

## 9. Flush & Verify Traces

Force-flush any remaining spans to LangSmith and confirm export.

In [ ]:
provider.force_flush()
print("✅ All spans flushed to LangSmith")
print(f"\n🔗 Check your traces at: https://smith.langchain.com/")
print(f"   Look for service: '{os.environ['OTEL_SERVICE_NAME']}'")
print(f"   Root span: 'notebook.end_to_end_test'")

## 10. Cleanup

In [ ]:
provider.shutdown()
print("✅ TracerProvider shut down. Done!")